### Medicine label reading assistant

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from rapidocr_onnxruntime import RapidOCR

label_image = Image.new("RGB", (900, 380), "white")
draw = ImageDraw.Draw(label_image)
font = ImageFont.load_default(size=40)
for index, line in enumerate(["SAMPLE MEDICINE LABEL", "MEDICINE: EXAMPLE", "STRENGTH: 100 MG", "EXPIRY: 2030-12", "NOT FOR MEDICAL USE"]):
    draw.text((30, 25 + index * 65), line, font=font, fill="black")

ocr = RapidOCR()

def read_medicine_label(image_rgb):
    image_bgr = cv2.cvtColor(np.asarray(image_rgb.convert("RGB")), cv2.COLOR_RGB2BGR)
    result, _ = ocr(image_bgr)
    if not result:
        print("No text detected. Try a clearer, well-lit image.")
        return []
    lines = []
    for _, text, confidence in result:
        lines.append(text)
        print(f"{text} (OCR confidence: {confidence:.2f})")
    print("\nOCR may be inaccurate. Verify the original label with a pharmacist; do not use this output to decide what medicine or dose to take.")
    return lines

plt.figure(figsize=(10, 4))
plt.imshow(label_image)
plt.axis("off")
plt.show()
label_lines = read_medicine_label(label_image)

### Facial-expression-aware assistant

In [ ]:
from pathlib import Path
from fer.fer import FER

sample_path = Path("astronaut.png")
if not sample_path.is_file():
    sample_path = Path("A9") / "astronaut.png"
with Image.open(sample_path) as image:
    face_image = image.convert("RGB")
detector = FER()

def expression_assistant(image_rgb):
    image_bgr = cv2.cvtColor(np.asarray(image_rgb.convert("RGB")), cv2.COLOR_RGB2BGR)
    faces = detector.detect_emotions(image_bgr)
    if not faces:
        print("No face detected. Try a clear, front-facing image with permission.")
        return []
    for index, face in enumerate(faces, start=1):
        label, score = max(face["emotions"].items(), key=lambda item: item[1])
        print(f"Face {index}: predicted expression = {label}; model score = {score:.2f}")
        if score < 0.5:
            print("The expression prediction is uncertain.")
            prompt = "How are you feeling? Let me know how I can help."
        elif label in {"sad", "angry", "fear", "disgust"}:
            prompt = "Would you like to talk, take a break, or do something else?"
        elif label in {"happy", "surprise"}:
            prompt = "Would you like to share what's on your mind or continue with a task?"
        else:
            prompt = "How can I help you today?"
        print(f"Assistant: {prompt}")
    print("Expression predictions are approximate and do not determine a person's actual feelings or mental state.")
    return faces

plt.figure(figsize=(5, 5))
plt.imshow(face_image)
plt.axis("off")
plt.show()
face_results = expression_assistant(face_image)

### Use your own images

In [ ]:
label_path = ""
face_path = ""

if label_path:
    with Image.open(label_path) as image:
        read_medicine_label(image)
if face_path:
    with Image.open(face_path) as image:
        expression_assistant(image)
